# 0825_lsw_007_cleanlab_confident_learning

004의 라벨 클렌징(피처가 완전 동일한데 라벨이 다른 행만 제거)과 005의 라벨 보정은 **아주 좁은
정의의 모순만** 잡았다. cleanlab의 confident learning은 이보다 훨씬 넓게 "모델이 강하게 반대하는
라벨"을 통계적으로 찾아낸다(로드맵 Phase 3-1). 이번 노트북은:

1. 검사유형별 Train에 대해 out-of-fold 예측 확률(cross-validation)을 구하고, `cleanlab.filter.
   find_label_issues`로 라벨 이슈 후보를 찾는다.
2. 이슈 행을 제거한 뒤 baseline과 비교한다(ΔTN/ΔFN, 총비용 — 규칙대로).
3. **004에서 이미 검증된 유형별 최고 기법(undersample/adasyn/smote 등) 위에 cleanlab 클렌징을
   추가로 얹어서**, baseline이 아니라 **현재 최고 기법 대비** 더 개선되는지 확인한다(최근 합의한
   "현재 최고 유지 + 개선분 탐색" 방식).

Validation/Test는 항상 원본 그대로 둔다(클렌징은 Train에만 적용, 데이터 누수 방지).


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from cleanlab.filter import find_label_issues
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_007_cleanlab_confident_learning"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


C:\Users\ajou\Desktop\manufacturing_AI\siemens_aoi_ML_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


experiment: 0825_lsw_007_cleanlab_confident_learning


## 2. 데이터 로딩·전처리 (003~006과 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time


## 3. 평가 함수 (003~006과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model(scale_pos_weight=1.0):
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    )


def resample_train(technique, X_train, y_train, n_pos):
    k_neighbors = max(1, min(5, n_pos - 1))
    if technique == "smote":
        return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors).fit_resample(X_train, y_train)
    if technique == "adasyn":
        try:
            return ADASYN(random_state=RANDOM_STATE, n_neighbors=k_neighbors).fit_resample(X_train, y_train)
        except ValueError:
            return X_train, y_train
    if technique == "undersample":
        return RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    return X_train, y_train


## 4. 검사유형별 subset 및 baseline

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df,
        "valid": type_valid_df,
        "test": type_test_df,
        "feature_columns": type_feature_columns,
    }


def fit_select_evaluate(train_df, valid_df, test_df, feature_columns, model):
    model.fit(train_df[feature_columns], train_df[TARGET])
    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
    return evaluate_at_threshold(test_df[TARGET], test_proba, threshold)


baseline_results = {}
for inspection_type, split in type_splits.items():
    baseline_results[inspection_type] = fit_select_evaluate(
        split["train"], split["valid"], split["test"], split["feature_columns"], build_model()
    )
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = "inspection_type"
baseline_df[["threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction"]]


,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction
inspection_type,,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,0.067265,0.165414,0.531920
1,0.000025,1869.0,8459.0,8.0,776.0,0.403144,0.010204,0.180964
2,0.000006,3492.0,14958.0,12.0,691.0,0.356827,0.017070,0.189268
3,0.000007,9700.0,20288.0,43.0,560.0,0.269568,0.071310,0.323463
4,0.000001,0.0,730.0,0.0,26.0,0.023284,0.000000,0.000000


## 5. 004 기법 전체(불균형 4종 + 라벨 클렌징) 재현 — 비용비율별 "현재 최고 기법" 확보

004/005에서 유형별 최고 기법에는 불균형 4기법뿐 아니라 **라벨 클렌징(모순 라벨 삭제)** 도
있었다(type0). 빠뜨리면 안 되므로 여기도 포함한다. 또한 1:10과 1:100에서 최고 기법이 다를 수
있으므로(예: type3은 1:10=adasyn, 1:100=smote) **비용비율마다 따로** 최고를 고른다 — 합산 비용
하나로 뭉뚱그리지 않는다.


In [5]:
techniques = ["class_weight", "smote", "adasyn", "undersample"]
resample_results = {"baseline": baseline_results}

for technique in techniques:
    per_type = {}
    for inspection_type, split in type_splits.items():
        train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
        feature_columns = split["feature_columns"]
        X_train, y_train = train_df[feature_columns], train_df[TARGET]
        n_pos = int((y_train == 1).sum())
        n_neg = int((y_train == 0).sum())

        if technique == "class_weight":
            X_res, y_res = X_train, y_train
            model = build_model(scale_pos_weight=n_neg / n_pos)
        else:
            X_res, y_res = resample_train(technique, X_train, y_train, n_pos)
            model = build_model()

        model.fit(X_res, y_res)
        valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
        threshold = select_threshold(valid_df[TARGET], valid_proba)
        test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
        per_type[inspection_type] = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
    resample_results[technique] = per_type

# 라벨 클렌징(모순 라벨 삭제, 004/005와 동일 로직) 추가
label_cleansing_per_type = {}
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]
    nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
    contradictory_mask = nunique_classes > 1
    train_clean_df = train_df.loc[~contradictory_mask]
    label_cleansing_per_type[inspection_type] = fit_select_evaluate(
        train_clean_df, valid_df, test_df, feature_columns, build_model()
    )
resample_results["label_cleansing"] = label_cleansing_per_type

current_best_technique = {"1:10": {}, "1:100": {}}
current_best_result = {"1:10": {}, "1:100": {}}
for scenario in ["1:10", "1:100"]:
    for inspection_type in sorted(clean_df["inspection_type"].unique()):
        costs = {name: results[inspection_type][f"total_cost_{scenario}"] for name, results in resample_results.items()}
        best_name = min(costs, key=costs.get)
        current_best_technique[scenario][inspection_type] = best_name
        current_best_result[scenario][inspection_type] = resample_results[best_name][inspection_type]

pd.DataFrame(
    {
        "최고기법(1:10)": current_best_technique["1:10"],
        "총비용(1:10)": {t: r["total_cost_1:10"] for t, r in current_best_result["1:10"].items()},
        "최고기법(1:100)": current_best_technique["1:100"],
        "총비용(1:100)": {t: r["total_cost_1:100"] for t, r in current_best_result["1:100"].items()},
    }
)


,최고기법(1:10),총비용(1:10),최고기법(1:100),총비용(1:100)
0,label_cleansing,5678,label_cleansing,9818
1,undersample,6209,undersample,7289
2,undersample,10541,undersample,14411
3,adasyn,17515,smote,22456
4,undersample,715,undersample,715


## 6. cleanlab confident learning으로 라벨 이슈 탐지

검사유형별 Train에 대해 StratifiedKFold out-of-fold 예측 확률을 구하고,
`cleanlab.filter.find_label_issues`로 이슈 후보를 찾는다.

In [6]:
label_issue_masks = {}
n_flagged_by_type = {}

for inspection_type, split in type_splits.items():
    train_df = split["train"]
    feature_columns = split["feature_columns"]
    X_train, y_train = train_df[feature_columns], train_df[TARGET].to_numpy()

    n_pos = int((y_train == 1).sum())
    n_splits = min(5, max(2, n_pos))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof_proba = cross_val_predict(build_model(), X_train, y_train, cv=skf, method="predict_proba", n_jobs=-1)
    issue_mask = find_label_issues(labels=y_train, pred_probs=oof_proba)

    label_issue_masks[inspection_type] = issue_mask
    n_flagged_by_type[inspection_type] = int(issue_mask.sum())
    print(f"type {inspection_type}: n_splits={n_splits}, train_pos={n_pos}, flagged={issue_mask.sum()} / {len(y_train)}")


type 0: n_splits=5, train_pos=57, flagged=52 / 41961


type 1: n_splits=5, train_pos=555, flagged=152 / 34041


type 2: n_splits=5, train_pos=579, flagged=63 / 74910


type 3: n_splits=5, train_pos=620, flagged=79 / 80792


type 4: n_splits=5, train_pos=13, flagged=7 / 3518


## 7. cleanlab 클렌징 단독 적용 (baseline 대비)

In [7]:
cleanlab_alone_results = {}
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]
    issue_mask = label_issue_masks[inspection_type]

    train_clean_df = train_df.loc[~issue_mask]
    cleanlab_alone_results[inspection_type] = fit_select_evaluate(
        train_clean_df, valid_df, test_df, feature_columns, build_model()
    )

cleanlab_alone_df = pd.DataFrame(cleanlab_alone_results).T
cleanlab_alone_df.index.name = "inspection_type"

compare_vs_baseline_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    base = baseline_results[inspection_type]
    clean = cleanlab_alone_results[inspection_type]
    compare_vs_baseline_rows.append(
        {
            "inspection_type": inspection_type,
            "n_flagged": n_flagged_by_type[inspection_type],
            "baseline_TN": base["tn"], "baseline_FN": base["fn"],
            "cleanlab_TN": clean["tn"], "cleanlab_FN": clean["fn"],
            "ΔTN": clean["tn"] - base["tn"], "ΔFN": clean["fn"] - base["fn"],
            "baseline_총비용(1:10)": base["total_cost_1:10"], "cleanlab_총비용(1:10)": clean["total_cost_1:10"],
            "baseline_총비용(1:100)": base["total_cost_1:100"], "cleanlab_총비용(1:100)": clean["total_cost_1:100"],
        }
    )
pd.DataFrame(compare_vs_baseline_rows).set_index("inspection_type")


,n_flagged,baseline_TN,baseline_FN,cleanlab_TN,cleanlab_FN,ΔTN,ΔFN,baseline_총비용(1:10),cleanlab_총비용(1:10),baseline_총비용(1:100),cleanlab_총비용(1:100)
inspection_type,,,,,,,,,,,
0,52,8857,22,779,11,-8078,-11,8014,15982,9994,16972
1,152,1869,8,3393,15,1524,7,8539,7085,9259,8435
2,63,3492,12,70,0,-3422,-12,15078,18380,16158,18380
3,79,9700,43,6252,28,-3448,-15,20718,24016,24588,26536
4,7,0,0,5,0,5,0,730,725,730,725


## 8. cleanlab 클렌징 + 기존 기법 전부 조합 (비용비율별 "현재 최고" 대비)

cleanlab으로 정제한 Train 위에서 기존 5기법(baseline/class_weight/smote/adasyn/undersample/
label_cleansing)을 전부 다시 적용해보고, 비용비율마다 그 안에서 최고를 골라 5절의 "현재 최고"와
비교한다. 특정 기법 하나만 고정해서 얹지 않는 이유: cleanlab이 데이터를 바꿔놓으면 원래 최고였던
기법이 더 이상 최고가 아닐 수 있어서다.

In [8]:
def fit_technique_on_train(train_df, valid_df, test_df, feature_columns, technique):
    if technique == "label_cleansing":
        nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
        train_df = train_df.loc[~(nunique_classes > 1)]

    X_train, y_train = train_df[feature_columns], train_df[TARGET]
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    if technique in ("baseline", "label_cleansing"):
        model = build_model()
        X_res, y_res = X_train, y_train
    elif technique == "class_weight":
        model = build_model(scale_pos_weight=n_neg / max(n_pos, 1))
        X_res, y_res = X_train, y_train
    else:
        model = build_model()
        X_res, y_res = resample_train(technique, X_train, y_train, n_pos)

    return fit_select_evaluate_prefit(X_res, y_res, valid_df, test_df, feature_columns, model)


def fit_select_evaluate_prefit(X_res, y_res, valid_df, test_df, feature_columns, model):
    model.fit(X_res, y_res)
    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
    return evaluate_at_threshold(test_df[TARGET], test_proba, threshold)


all_technique_names = ["baseline", "class_weight", "smote", "adasyn", "undersample", "label_cleansing"]
cleanlab_plus_all = {}
for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]
    issue_mask = label_issue_masks[inspection_type]
    train_clean_df = train_df.loc[~issue_mask]

    cleanlab_plus_all[inspection_type] = {
        technique: fit_technique_on_train(train_clean_df, valid_df, test_df, feature_columns, technique)
        for technique in all_technique_names
    }

cleanlab_best_technique = {"1:10": {}, "1:100": {}}
cleanlab_best_result = {"1:10": {}, "1:100": {}}
for scenario in ["1:10", "1:100"]:
    for inspection_type in sorted(clean_df["inspection_type"].unique()):
        costs = {name: r[f"total_cost_{scenario}"] for name, r in cleanlab_plus_all[inspection_type].items()}
        best_name = min(costs, key=costs.get)
        cleanlab_best_technique[scenario][inspection_type] = best_name
        cleanlab_best_result[scenario][inspection_type] = cleanlab_plus_all[inspection_type][best_name]

compare_vs_best_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    for scenario in ["1:10", "1:100"]:
        best = current_best_result[scenario][inspection_type]
        combo = cleanlab_best_result[scenario][inspection_type]
        compare_vs_best_rows.append(
            {
                "inspection_type": inspection_type,
                "비용비율": scenario,
                "현재_최고_기법(cleanlab 없음)": current_best_technique[scenario][inspection_type],
                "cleanlab+최고_기법": cleanlab_best_technique[scenario][inspection_type],
                "n_flagged": n_flagged_by_type[inspection_type],
                "현재최고_TN": best["tn"], "현재최고_FN": best["fn"],
                "cleanlab+최고_TN": combo["tn"], "cleanlab+최고_FN": combo["fn"],
                "ΔTN(현재최고 대비)": combo["tn"] - best["tn"], "ΔFN(현재최고 대비)": combo["fn"] - best["fn"],
                "현재최고_총비용": best[f"total_cost_{scenario}"],
                "cleanlab+최고_총비용": combo[f"total_cost_{scenario}"],
                "개선": combo[f"total_cost_{scenario}"] - best[f"total_cost_{scenario}"],
            }
        )
compare_vs_best_df = pd.DataFrame(compare_vs_best_rows).set_index(["inspection_type", "비용비율"])
compare_vs_best_df


현재_최고_기법(cleanlab 없음) cleanlab+최고_기법  n_flagged  \
inspection_type 비용비율                                                    
0               1:10        label_cleansing         adasyn         52   
                1:100       label_cleansing          smote         52   
1               1:10            undersample    undersample        152   
                1:100           undersample          smote        152   
2               1:10            undersample    undersample         63   
                1:100           undersample    undersample         63   
3               1:10                 adasyn          smote         79   
                1:100                 smote          smote         79   
4               1:10            undersample    undersample          7   
                1:100           undersample    undersample          7   

                       현재최고_TN  현재최고_FN  cleanlab+최고_TN  cleanlab+최고_FN  \
inspection_type 비용비율                                                      
0               1:10     11433       46            9290              26   
                1:100    11433       46            9235              25   
1               1:10      4239       12            4647              28   
                1:100     4239       12            3873              19   
2               1:10      8339       43            6697              30   
                1:100     8339       43            6697              30   
3               1:10     13033       56           21290             136   
                1:100    10632       31           21290             136   
4               1:10        15        0              15               0   
                1:100       15        0              15               0   

                       ΔTN(현재최고 대비)  ΔFN(현재최고 대비)  현재최고_총비용  cleanlab+최고_총비용  \
inspection_type 비용비율                                                           
0               1:10          -2143           -20      5678             7621   
                1:100         -2198           -21      9818             9916   
1               1:10            408            16      6209             5961   
                1:100          -366             7      7289             8355   
2               1:10          -1642           -13     10541            12053   
                1:100         -1642           -13     14411            14753   
3               1:10           8257            80     17515            10058   
                1:100         10658           105     22456            22298   
4               1:10              0             0       715              715   
                1:100             0             0       715              715   

                         개선  
inspection_type 비용비율         
0               1:10   1943  
                1:100    98  
1               1:10   -248  
                1:100  1066  
2               1:10   1512  
                1:100   342  
3               1:10  -7457  
                1:100  -158  
4               1:10      0  
                1:100     0

## 9. 결론 및 다음 단계

### 현재 최고 기법 대비 결과 (8절, "현재 최고 유지 + 개선분 탐색" 방식)

| type | 비용비율 | 기존 최고 | 비용 | cleanlab+최고 조합 | 비용 | 개선 |
|---|---|---|---:|---|---:|---:|
| 0 | 1:10 | label_cleansing | 5,678 | +adasyn | 7,621 | **-1,943 악화** |
| 0 | 1:100 | label_cleansing | 9,818 | +smote | 9,916 | -98 악화(소폭) |
| 1 | 1:10 | undersample | 6,209 | +undersample | 5,961 | **+248 개선** |
| 1 | 1:100 | undersample | 7,289 | +smote | 8,355 | -1,066 악화 |
| 2 | 1:10 | undersample | 10,541 | +undersample | 12,053 | -1,512 악화 |
| 2 | 1:100 | undersample | 14,411 | +undersample | 14,753 | -342 악화 |
| **3** | **1:10** | adasyn | 17,515 | **+smote** | **10,058** | **+7,457 개선(43%↓)** |
| **3** | **1:100** | smote | 22,456 | **+smote** | **22,298** | **+158 개선** |
| 4 | 1:10/1:100 | undersample | 715 | +undersample | 715 | 변화 없음(표본 부족) |

**답: "라벨 노이즈/drift 관련해서 성능 올릴 방법이 없다"는 아니다.** type3에서 cleanlab
confident learning + smote 조합이 **두 비용비율 모두에서 기존 최고를 확실히 이긴다**(1:10에서
43% 비용 절감). type1은 1:10에서만 소폭 개선. 나머지(type0/2/4)는 기존 최고가 여전히 낫다.

### 왜 type3만 확실히 통했나 — type0/2가 나빠진 이유에 대한 해석

cleanlab은 Train 양성 57건(type0)처럼 표본이 작고 baseline PR-AUC가 낮은(0.067) 유형에서는
out-of-fold 확률 추정 자체가 불안정하다 — 5-fold면 fold당 양성이 10건 안팎이라, "이 라벨이
의심스럽다"는 판단의 근거가 빈약하다. 실제로 type0은 52/57건(양성 대부분)이 flagged됐는데, 이건
노이즈 제거라기보다 **약한 신호를 가진 진짜 양성까지 통계적으로 이상해 보여서 지워버렸을
가능성**이 크다(7절의 baseline 대비 비교에서 TN이 8,857→779로 폭락한 게 그 증거 — 모델이 훨씬
공격적으로 바뀜). type2(양성 579건, PR-AUC 0.357로 나쁘지 않음)도 나빠진 걸 보면 표본 크기만의
문제는 아니고, notes.md에서 확인한 **type2의 진짜 concept drift(판정기준 변화)** 가 cleanlab
입장에서도 "일관성 없는 라벨"로 잡혀서, 실제로는 지우면 안 되는 정당한 최근 신호까지 같이
지워졌을 수 있다 — 004/005에서 이미 type2의 모순 라벨을 건드리면(삭제든 보정이든) 항상 손해였던
것과 같은 패턴이다.

type3(양성 620건, baseline PR-AUC 0.270)은 표본도 충분하고 type2 같은 강한 concept drift
신호도 없어서, cleanlab이 진짜 노이즈만 상대적으로 깨끗하게 골라낸 것으로 보인다.

### 검사유형별 최적 조합 갱신 (003~007 종합)

| type | 1:10 최적 | 1:100 최적 | 비고 |
|---|---|---|---|
| 0 | label_cleansing | label_cleansing | cleanlab은 악화 — 표본 부족(양성 57건)으로 신뢰 불가 |
| 1 | undersample (cleanlab+undersample과 근소, 실무 편의상 기존 유지 권장) | undersample | 1:10만 소폭 개선이라 굳이 바꿀 실익 적음 |
| 2 | undersample | undersample | cleanlab 악화 — concept drift(type2 특유)와 충돌 추정 |
| **3** | **cleanlab + smote** | **cleanlab + smote** | **새 챔피언 — 두 시나리오 모두 확실한 개선** |
| 4 | undersample | undersample | 표본 부족으로 cleanlab 무의미 |

### 다음 단계

1. type3에 한해 cleanlab+smote를 채택하고 `docs/model_val.md`에 반영을 검토한다.
2. type0/2처럼 표본이 작거나 concept drift가 강한 유형에는 confident learning류 기법을
   무분별하게 적용하면 안 된다는 게 이번의 핵심 교훈 — "라벨 정제 기법 자체가 유형의 특성(표본
   크기, drift 강도)에 좌우된다."
3. 다음: 사용자가 제안한 "치팅 아닌 버전"의 임계값 재조정(과거 확정 구간의 관측 불량률로 다음
   구간 임계값을 조정) 실험으로 이어간다.
